In [27]:
import yfinance as yf
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [24]:
def save_10_year_single_stock_data_to_csv(ticker: str, per: str = "10y") -> pd.DataFrame:
    data = yf.download(
    ticker,
    period=per,
    interval="1d",
    auto_adjust=True,
    progress=False
    )

    if isinstance(data.columns, pd.MultiIndex):
        data.columns = data.columns.get_level_values(0)

    data_dir = Path("data")
    data_dir.mkdir(exist_ok=True)

    filename = data_dir / f"{ticker}_{per}_auto_adjusted.csv"
    data.to_csv(filename)

    return data

def create_returns_and_save(
    df: pd.DataFrame,
    ticker: str,
    period: str = "10y",
    data_folder: str = "data"
    ) -> pd.DataFrame:
    """
    Create a derived dataset containing daily arithmetic and logarithmic returns,
    save it as a new CSV file in the specified data folder, and return the new DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        Input price DataFrame containing at minimum a 'Close' column.
        The index is expected to be a DatetimeIndex.
    
    ticker : str
        Stock ticker symbol used for naming the output file.

    period : str, default="10y"
        Period label used in the output filename.

    data_folder : str, default="data"
        Directory where the resulting CSV file will be stored.
        The folder will be created if it does not exist.

    Returns
    -------
    pd.DataFrame
        A new DataFrame containing:
        - daily_return : arithmetic daily return (percentage change)
        - log_return   : logarithmic daily return

        The first row is removed due to undefined return values.

    Notes
    -----
    - Arithmetic returns are computed as:
        R_t = (P_t / P_{t-1}) - 1

    - Logarithmic returns are computed as:
        r_t = ln(P_t / P_{t-1})

    - The original DataFrame is not modified.
    - The output file is named:
        {ticker}_{period}_with_returns.csv
    """
    
    new_df = df.copy()

    new_df["daily_return"] = new_df["Close"].pct_change()
    new_df["log_return"] = np.log(
        new_df["Close"] / new_df["Close"].shift(1)
    )

    new_df = new_df.dropna()

    data_path = Path(data_folder)
    data_path.mkdir(exist_ok=True)

    filename = data_path / f"{ticker}_{period}_with_returns.csv"
    new_df.to_csv(filename)

    return new_df

In [22]:
ticker = "BAC"

data = save_10_year_single_stock_data_to_csv(ticker)
data

Price,Close,High,Low,Open,Volume
Date,,,,,
2016-02-19,9.742739,9.831090,9.622259,9.815026,121045600
2016-02-22,10.072049,10.072049,9.742740,9.742740,88865500
2016-02-23,9.766837,10.023859,9.718646,10.015828,104498700
2016-02-24,9.742739,9.758802,9.357205,9.606196,159552800
2016-02-25,9.895347,9.903379,9.710612,9.750772,103054600
...,...,...,...,...,...
2026-02-11,53.849998,56.110001,53.240002,55.970001,48973500
2026-02-12,52.520000,53.939999,51.790001,53.910000,55481600
2026-02-13,52.549999,52.820000,51.439999,51.910000,31772100


In [28]:
AAPL_data = pd.read_csv(
    "data/AAPL_10y_auto_adjusted.csv",
    index_col=0,
    parse_dates=True,
    )

JNJ_data = pd.read_csv(
    "data/JNJ_10y_auto_adjusted.csv",
    index_col=0,
    parse_dates=True,
    )
    

JNJ_data, AAPL_data

return_data_JNJ = create_returns_and_save(JNJ_data, 'JNJ')
return_data_AAPL = create_returns_and_save(AAPL_data, 'APPL')

In [26]:
fig = go.Figure(
    data=[
        go.Candlestick(
            x=AAPL_data.index,
            open=AAPL_data["Open"],
            high=AAPL_data["High"],
            low=AAPL_data["Low"],
            close=AAPL_data["Close"],
            name="AAPL"
        )
    ]
)

fig.update_layout(
    title="AAPL Daily Candlestick",
    xaxis_title="Date",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False
)

fig.show()